## Processing Tool for Swissleague HF Overal Ranking

The needed Imports:


In [ ]:
import pandas as pd
import json
from pathlib import Path
!pip install reportlab
from reportlab.platypus import SimpleDocTemplate, Table, TableStyle, Flowable, Paragraph, Spacer
from reportlab.lib.pagesizes import A4
from reportlab.lib import colors
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import Image
from reportlab.lib.utils import ImageReader
from reportlab.lib.styles import ParagraphStyle
from reportlab.lib.colors import grey

Add Drive to the Notebook to access data

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


Add path to excel files

In [ ]:
drive_path = Path('/content/drive')
files_path = drive_path / 'MyDrive/Swissleague'

Specify Titels and Paths to Data

In [ ]:
# Output Titles
overall_title = "Swissleague Hike and Fly Overall Ranking"
women_title = "Swissleague Hike and Fly Female Ranking"
men_title = "Swissleague Hike and Fly Male Ranking"

# Datapath to the swissleague logo for the PDF
logo_path = files_path / "swisscup_hf_farbe.png"

# Kleingedrucktes
info = "For feedbacks, please contact: sport@shv-fsvl.ch"

# Competitions
jhf_title = "Jura Hike and Fly"

# Datapath to preprocessed JSON
json_path = files_path / "swissleague_data_2025.json"
competition_path = files_path / "swissleague_competitions_2025.json"

# Datapaths to competions excel
jhf_path = "./jhf_2025.xlsx"

# specify which Excel files shall be processed and corresponding json keys
excel_files = [jhf_path]
json_keys = ["jhf"]
titles = [jhf_title]
physical_array = [False]

Load the json file

In [ ]:
json_data = {}
# check if json already exists:
try:
    with open(json_path, "r") as f:
        json_data = json.load(f)
except FileNotFoundError:
    print("Eval JSON file does not exist yet.")
    json_data = {}

competition_json = {}
try:
    with open(competition_path, "r") as f:
        competition_json = json.load(f)
except FileNotFoundError:
    print("Competition JSON file does not exist yet.")
    competition_json = {}

    # "Fly Swissalps", "Flyback, Frutigen", "Trailfly", "Vercofly", "Beizenfliegen", "Belli in Fly", "Millets Cup"

for key, title, physical in zip(["swissalps", "flyback", "trailfly", "vercofly", "beizen", "belli", "millet"],["Fly Swissalps", "Flyback, Frutigen", "Trailfly", "Vercofly", "Beizenfliegen", "Belli in Fly", "Millets Cup"],[False, False, False, False, False, True, True]):
    competition_json[key] = {
        "title": title,
        "num_participants": 0,
        "physical": physical
    }

Functions used later in the script

In [ ]:
def print_name_discrepancy(json_data, row, key):
        print(f"Discrepancy found for civl_id {row['civl_id']}:")
        print(f"{key} in JSON: {json_data[str(row['civl_id'])][key]}, {key} in Excel: {row[key]}")

def check_civl_id_name_discrepancy(json_data, row):
      # check if there is a discrepancy between names
      if json_data[str(row['civl_id'])]['name'] != row['last_name']:
        print_name_discrepancy(json_data, row, 'name')
      if json_data[str(row['civl_id'])]['first_name'] != row['first_name']:
        print_name_discrepancy(json_data, row, 'first_name')

def add_points_to_data(df: pd.DataFrame):
    # Get number of Participants
    num_participants = len(df)

    # Formula: 100 - 100 * (rank - 1) / (num_participants -1)
    df['points'] = (100 - (100 * (df['rank'] - 1)) / (num_participants - 1)).clip(lower=1).round(2)

    return num_participants, df

def check_if_competion_already_exists(json_data, civil_id, competition_key):
    if competition_key in json_data[civil_id]:
      print(f"Competition {competition_key} already exists for civl_id {civil_id}")
      return True
    return False

def add_competition_data(json_data, data_row, competition_key):
    json_data[str(data_row['civl_id'])][competition_key] = {
        "rank": data_row['rank'],
        "points": data_row['points'],
        "counts": True
    }
    return json_data

def update_total_points(json_data, civil_id:str, new_points, new_comp_key):
    athletes_comp_keys = json_data[civil_id].keys()
    athletes_comp_keys = remove_athletes_data_from_comp_keys(athletes_comp_keys)
    if len(athletes_comp_keys) <= 4:
      total_points = json_data[civil_id][total_points] + new_points
      json_data[civil_id][total_points] = total_points

    else:
      # Find lowest points from previous comps
      lowest_points = new_points
      lowest_key = new_comp_key

      for comp_key in athletes_comp_keys:
          if json_data[civil_id][comp_key]['counts'] and json_data[civil_id][comp_key]['points'] < lowest_points:
            lowest_points = json_data[civil_id][comp_key]['points']
            lowest_key = comp_key

      # update remove lowest_competition from total points
      json_data[civil_id][lowest_key]['counts'] = False
      json_data[civil_id][total_points] += new_points - lowest_points
    return json_data

def remove_athletes_data_from_comp_keys(athletes_comp_keys):
      athletes_comp_keys = athletes_comp_keys.remove('name')
      athletes_comp_keys = athletes_comp_keys.remove('first_name')
      athletes_comp_keys = athletes_comp_keys.remove('gender')
      athletes_comp_keys = athletes_comp_keys.remove('total_points')
      return athletes_comp_keys

def add_athlete_data(json_data, data_row):
    json_data[str(data_row['civl_id'])] = {
        "name": data_row['last_name'],
        "first_name": data_row['first_name'],
        "gender": data_row['gender'],
        "total_points": data_row['points']
    }
    return json_data

def add_data_to_json(json_data: dict, df: pd.DataFrame, competition_key: str):
    # add data to json
    for _, row in df.iterrows():
        # check if civl_id already exists
        if str(row['civl_id']) in json_data:
            # check if there is name discrepancy for manual update
            check_civl_id_name_discrepancy(json_data, row)

        else:
            # add athletes data
            json_data = add_athlete_data(json_data, row)

        # add the competion
        check_if_competion_already_exists(json_data, str(row['civl_id']), competition_key)
        json_data = add_competition_data(json_data, row, competition_key)

    return json_data

Now load the excel files and add the data to the JSON

In [ ]:
for titel, competition_key, excel_file, physical in zip(titles, json_keys, excel_files, physical_array):
    # Load the Excel file
    excel_path = files_path / excel_file
    df = pd.read_excel(excel_path)

    # Select only the necessary columns and rename for consistency
    df = df[['Rank', 'First Name', 'Last Name', 'Gender', 'CIVL ID']]
    df.columns = ['rank', 'first_name', 'last_name', 'gender', 'civl_id']

    # Evaluting the Points received based on ranking
    num_part, df = add_points_to_data(df)

    # add data to json
    json_data = add_data_to_json(json_data, df, competition_key)

    # Write competition info into a competition file
    competition_json[competition_key] = {
        "title": titel,
        "num_participants": num_part,
        "physical": physical
    }

# Save to JSON files
with open(json_path, "w") as f:
    json.dump(json_data, f, indent=4)

with open(competition_path, "w") as f:
    json.dump(competition_json, f, indent=4)

Competition jhf already exists for civl_id 1414
Competition jhf already exists for civl_id 85387
Competition jhf already exists for civl_id 35556
Competition jhf already exists for civl_id 87138
Competition jhf already exists for civl_id 73646
Competition jhf already exists for civl_id 90734
Competition jhf already exists for civl_id 82484
Competition jhf already exists for civl_id 82160
Competition jhf already exists for civl_id 72442
Competition jhf already exists for civl_id 55941
Competition jhf already exists for civl_id 64916
Competition jhf already exists for civl_id 55154
Competition jhf already exists for civl_id 73663
Competition jhf already exists for civl_id 73648
Competition jhf already exists for civl_id 90602
Competition jhf already exists for civl_id 81365
Competition jhf already exists for civl_id 89383
Competition jhf already exists for civl_id 78576
Competition jhf already exists for civl_id 90617
Competition jhf already exists for civl_id 90605
Competition jhf alrea

### Generate PDFs for Publication

Functions needed for PDF generation

In [ ]:
def add_data_to_lists(overall_rows: list, gender_rows: list, athlete_data: dict, competitions: list, civil_id):
    row = {
        "name": f"{athlete_data['first_name']} {athlete_data['name']}",
        }
    for comp in competitions:
        try:
           row[comp] = athlete_data[comp]["points"]
        except KeyError:
           row[comp] = ""
    row["total_points"] = athlete_data["total_points"]
    row["civil_id"] = civil_id

    # add the data row to the lists
    overall_rows.append(row), gender_rows.append(row)

    return overall_rows, gender_rows


def generate_pandas_data_frames(json_data: dict, competitions: list):
    rows_female = []
    rows_male = []
    rows_overall = []
    for civl_id in json_data.keys():
        if json_data[civl_id]['gender'] == 'F':
            rows_overall, rows_female = add_data_to_lists(rows_overall, rows_female, json_data[civl_id], all_competitions, civl_id)
        else:
            rows_overall, rows_male = add_data_to_lists(rows_overall, rows_male, json_data[civl_id], all_competitions, civl_id)

    df_female = pd.DataFrame(rows_female)
    df_male = pd.DataFrame(rows_male)
    df_overall = pd.DataFrame(rows_overall)

    return df_female, df_male, df_overall


def extract_competition_name_list(competition_json: dict):
    names = []
    for comp in competition_json.keys():
        names.append(competition_json[comp]["title"])
    return names



class RotatedHeader(Flowable):
    def __init__(self, text, width=40, height=60, fontSize=10):
        super().__init__()
        self.text = text
        self.width = width
        self.height = height
        self.fontSize = fontSize

    def draw(self):
        self.canv.saveState()
        self.canv.setFont("Helvetica-Bold", self.fontSize)

        # Move origin to the bottom center of the cell
        self.canv.translate(0, 0)

        # Rotate around the new origin
        self.canv.rotate(90)

        # Draw string so it's centered and touches bottom
        text_width = self.canv.stringWidth(self.text, "Helvetica", self.fontSize)
        self.canv.drawString(-43 , -24, self.text) # the first number aligns up/down, smaller -> down, secon number moves left/rigth, smaller -> right, but they also influence each other

        self.canv.restoreState()

    def wrap(self, availWidth, availHeight):
        return self.width, self.height




def generate_single_pdf(json_data: dict, competition_json: dict, df: pd.DataFrame, titel: str, path: str):
    doc = SimpleDocTemplate(str(path), pagesize=A4, leftMargin=20, rightMargin=20, topMargin=30, bottomMargin=20)

    styles = getSampleStyleSheet()
    story = []

    # Load and resize image with preserved aspect ratio
    img_reader = ImageReader(logo_path)
    orig_width, orig_height = img_reader.getSize()
    target_width = 100
    aspect_ratio = orig_height / orig_width
    target_height = target_width * aspect_ratio
    logo = Image(logo_path, width=target_width, height=target_height)

    # Create title paragraph
    title_para = Paragraph(f"<b>{titel}</b>", styles["Title"])

    # Create a 1-row, 2-column table with logo and title
    title_table = Table([[title_para, logo]], colWidths=[400, target_width + 10])  # Adjust second colWidth as needed
    title_table.setStyle(TableStyle([
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),
        ("ALIGN", (1, 0), (1, 0), "LEFT"),  # Align title left if desired
        ("LEFTPADDING", (0, 0), (-1, -1), 0),
        ("RIGHTPADDING", (0, 0), (-1, -1), 0),
        ("TOPPADDING", (0, 0), (-1, -1), 0),
        ("BOTTOMPADDING", (0, 0), (-1, -1), 0),
    ]))

    # Add to story
    story.append(title_table)
    story.append(Spacer(1, 12))

    # add infos, so called "Kleingedrucktes"
    # Create a custom small grey text style
    info_style = ParagraphStyle(
        name="InfoStyle",
        parent=styles["Normal"],
        fontSize=8,
        textColor=grey,
        spaceBefore=2,
        spaceAfter=2,
    )

    story.append(Paragraph(info, info_style))

    # Prepare table header
    competitions = extract_competition_name_list(competition_json)
    raw_header = ["Rank", "Name"] + competitions + ["Total Points"]
    header = []

    for i, col in enumerate(raw_header):
         header.append(RotatedHeader(col))

    # Prepare table rows
    table_data = [header]
    highlight_cells = []
    previous_points = 0
    pervious_rank = 0
    for rank, (idx, row) in enumerate(df.iterrows(), 1):
        if row["total_points"] == previous_points:
            athletes_true_rank = pervious_rank
        else:
            athletes_true_rank = rank
            previous_points = row["total_points"]
            pervious_rank = rank
        row_data = [str(athletes_true_rank), row["name"]]

        for offset, comp_key in enumerate(competition_json.keys()):
            row_data.append(str(row[comp_key]))

            try:
              # If this comp counts for the athlete, store (row_index, col_index) for highlighting
              if json_data[row["civil_id"]][comp_key]["counts"]:
                  table_row_idx = len(table_data)  # current row index in table
                  table_col_idx = 2 + offset     # offset by Rank and Name columns
                  highlight_cells.append((table_row_idx, table_col_idx))
            except KeyError:
              continue

        row_data.append(str(row.get("total_points", "")))
        table_data.append(row_data)
    # Build the table
    col_widths = [15, 135] + [30] * len(competitions) + [30]
    row_heights = [155] + [None] * (len(table_data) - 1)
    table = Table(table_data, colWidths=col_widths, rowHeights=row_heights, repeatRows=1)

    style = TableStyle([
        # Basic grid
        ("GRID", (0, 0), (-1, -1), 0.5, colors.black),

        # Header styling
        ("ALIGN", (0, 0), (-1, 0), "CENTER"),
        ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
        ("BACKGROUND", (0, 0), (-1, 0), colors.lightgrey),

        # Body alignment
        ("ALIGN", (0, 1), (0, -1), "CENTER"),  # Rank
        ("ALIGN", (1, 1), (1, -1), "LEFT"),    # Name
        ("ALIGN", (2, 1), (-1, -1), "CENTER"), # Competitions + Total Points
        ("VALIGN", (0, 0), (-1, -1), "MIDDLE"),

        # Bold outer border
        ("BOX", (0, 0), (-1, -1), 1.5, colors.black),

        # Bold line after header row
        ("LINEBELOW", (0, 0), (-1, 0), 1.5, colors.black),

        # Bold vertical line after second column (Name)
        ("LINEAFTER", (1, 0), (1, -1), 1.5, colors.black),

        # Bold vertical line before last column (Total Points)
        ("LINEBEFORE", (-1, 0), (-1, -1), 1.5, colors.black),
    ])

    highlight_color = colors.Color(red=172/255, green=220/255, blue=149/255, alpha = 0.6)


    # Light grey background for alternating rows (excluding header)
    for i in range(1, len(table_data)):
        if i % 2 == 0:  # Even-numbered row (index starts at 0)
            style.add("BACKGROUND", (0, i), (-1, i), colors.whitesmoke)

    for r, c in highlight_cells:
        style.add("BACKGROUND", (c, r), (c, r), highlight_color)

    table.setStyle(style)

    story.append(table)
    doc.build(story)


Run the PDF generation

In [ ]:
all_competitions = competition_json.keys()

df_female, df_male, df_overall = generate_pandas_data_frames(json_data, all_competitions)

# Sort by total points descending
df_female.sort_values("total_points", ascending=False, inplace=True)
df_male.sort_values("total_points", ascending=False, inplace=True)
df_overall.sort_values("total_points", ascending=False, inplace=True)

# generate the PDFs
generate_single_pdf(json_data, competition_json, df_female, women_title, files_path / "female_ranking_portrait.pdf")
generate_single_pdf(json_data, competition_json, df_male, men_title, files_path / "male_ranking_portrait.pdf")
generate_single_pdf(json_data, competition_json, df_overall, overall_title, files_path / "overall_ranking_portrait.pdf")

